# Step 3 — Value Matching (Synonyms + Entity Matching)

**Mistplay Genie Workshop**

In Step 2 the space couldn't answer *"revenue from PayPal"* — the data stores the code `PYPL`, and *"California"* — the data stores `CA`. Comments describe the columns but don't map business words to stored values.

**Value matching** fixes this. On each code column we add:
- **descriptions** — what the codes mean,
- **synonyms** — alternate words users type ("credit card", "Cali"),
- **entity matching** — Genie indexes the column's distinct *values* so it can resolve user text to a stored code.

This is a **column-level** configuration (`column_configs`) on the tables already in the space — so we **round-trip**: fetch the space, add the configs, patch it back.

> **API vs UI:** `column_configs` (synonyms / entity matching) are fully settable through the REST API — no builder shortcut, so we edit the fetched JSON and PATCH. No UI step required.

In [ ]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.questions
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.questions):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig, setup_widgets
from workshop_lib import genie_builder as gb
from workshop_lib import questions
setup_widgets(dbutils)
cfg = WorkshopConfig.from_widgets(dbutils)
client = gb.GenieClient()
space_id = gb.require_space_id(spark, cfg)   # created in Step 2
print("Editing Genie space:", space_id)
print(client.space_url(space_id))

## 1. Define the value-matching configs

- **fact_transactions.payment_method** — decode the codes + synonyms + entity matching.
- **fact_transactions.currency** — decode USD / CAD.
- **fact_transactions.status** — decode + synonyms ("refund", "paid").
- **dim_geography.state_code** — entity matching + synonyms so "California"/"Cali" resolve to `CA`.

`gb.column_config(...)` builds each entry.

In [ ]:
txn = cfg.gold("fact_transactions")
geo = cfg.gold("dim_geography")

txn_configs = [
    gb.column_config(
        "payment_method",
        description="Payment method. Codes: CC=Credit Card, PYPL=PayPal, GIFT=Gift Card, DEBIT=Debit Card.",
        synonyms=["credit card", "paypal", "gift card", "debit card", "payment type"],
        entity_matching=True,
    ),
    gb.column_config(
        "currency",
        description="Currency. USD=US Dollars, CAD=Canadian Dollars.",
        synonyms=["us dollars", "canadian dollars"],
        entity_matching=True,
    ),
    gb.column_config(
        "status",
        description="Transaction status. COMPLETED=paid, REFUNDED=refunded, FAILED=failed, PENDING=not settled.",
        synonyms=["refund", "refunds", "paid", "settled", "failed payment"],
        entity_matching=True,
    ),
]
geo_configs = [
    gb.column_config(
        "state_code",
        description="Two-letter state/province code, e.g. CA=California, NY=New York, ON=Ontario.",
        synonyms=["state", "province", "cali"],
        entity_matching=True,
    ),
]
print("configs ready:", len(txn_configs), "on transactions,", len(geo_configs), "on geography")

## 2. Round-trip: fetch → add configs → patch

In [ ]:
builder = client.load_builder(space_id)          # fetch current definition
builder.set_column_configs(txn, txn_configs)      # attach value matching
builder.set_column_configs(geo, geo_configs)
print(builder.summary())

client.patch_space(
    space_id,
    title=cfg.genie_space_name,
    description="Mistplay Genie Workshop — with value matching.",
    warehouse_id=cfg.warehouse_id,
    serialized_space=builder.to_serialized(),
)
print("Patched.")

## 3. Verify the configs landed

In [ ]:
check = client.load_builder(space_id).to_dict()
for t in check["data_sources"]["tables"]:
    if t.get("column_configs"):
        cols = [c["column_name"] for c in t["column_configs"]]
        print(t["identifier"].split(".")[-1], "->", cols)

## 4. Try it — the two question sets

In [ ]:
print(questions.render(3))

## Recap & what's next

Genie now speaks the business's language for coded values. But it still can't compute governed **measures** — an on-time delivery *rate*, a gross *margin %* — consistently, because those are formulas spanning joined tables.

**→ Step 4** builds a **metric view** with measures, relationships, joins, and filters.